In [21]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_benchmark.py"
github_token = None

# Drive remains the home for images and benchmark outputs.
drive.mount("/content/drive")


def run_git(args, check=True):
    """Run git and print useful stderr without leaking the optional token."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(github_token, "***") if github_token else result.stdout
    stderr = result.stderr.replace(github_token, "***") if github_token else result.stderr
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        hint = ""
        if "403" in stderr or "Write access to repository not granted" in stderr:
            hint = (
                " The GITHUB_TOKEN secret exists, but GitHub rejected it. "
                "Create a token that has read access to this private repository "
                "and permission to read repository contents."
            )
        raise RuntimeError(f"Git command failed with exit code {result.returncode}: {args[:2]}.{hint}")
    return result


# Colab does not automatically check out the full GitHub repo when opening a notebook.
# For this private repo, add a Colab secret named GITHUB_TOKEN only if unauthenticated clone fails.
if (REPO_DIR / ".git").exists():
    pull_result = run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    if pull_result.returncode != 0:
        print("WARNING: git pull failed; continuing with the existing Colab checkout.")
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Restart the Colab runtime or remove that folder, then rerun this setup cell."
        )

    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs a Colab secret named GITHUB_TOKEN "
            "so the Colab runtime can clone src/radle_benchmark.py. "
            "Add the secret, restart or rerun this setup cell, then continue."
        )

    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")
    run_git(["git", "clone", clone_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Benchmark module not found after setup: {MODULE_PATH}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/RadLE_CRASH_Lab'...



In [22]:
# ==========================================
# 2. API CLIENT + BENCHMARK IMPORTS
# ==========================================
from openai import OpenAI

from radle_benchmark import create_scorer_view, run_benchmark

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    print("ERROR: Could not find OPENAI_API_KEY in Colab Secrets.")

try:
    os.environ["TEST_OPENROUTER_API_KEY"] = userdata.get("TEST_OPENROUTER_API_KEY")
except Exception:
    print("ERROR: Could not find TEST_OPENROUTER_API_KEY in Colab Secrets.")

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["TEST_OPENROUTER_API_KEY"],
)


In [23]:
# ==========================================
# 3. DRIVE PATHS + RUN CONFIG
# ==========================================
master_images_folder = "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE v2 Master Data"
final_output_csv = "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE_v2_test.csv"

# Recommended sequence: run TEST_LIMIT=1 first; if syntax/API plumbing passes, run TEST_LIMIT=5.
# Use TEST_LIMIT=None only when ready for the full benchmark.
TEST_LIMIT = 1


In [24]:
# ==========================================
# 4. RUN BENCHMARK
# ==========================================
df_final = run_benchmark(
    client=openrouter_client,
    openai_client=openai_client,
    image_folder=master_images_folder,
    output_csv=final_output_csv,
    test_limit=TEST_LIMIT,
)

print("\nFINAL DATAFRAME PREVIEW:")
from IPython.display import display

display(df_final.head())


TEST MODE: Running on first 1 cases only.
Processing 1 unique cases across 10 models...

[1/1] Case ID: 1 (1 images)
  -> gpt_5_5... OK (18.2s | 543 out / 1284 in | 29.8 tok/sec)
  -> claude_4_7_opus... OK (7.1s | 366 out / 1602 in | 51.5 tok/sec)
  -> gemini_3_1_pro... OK (6.6s | 441 out / 1412 in | 66.8 tok/sec)
  -> grok_4_20... OK (8.3s | 1311 out / 1230 in | 158.0 tok/sec)
  -> qwen_vl_max...
    FATAL ERROR (No Retry): Error code: 404 - {'error': {'message': 'No endpoints found for qwen/qwen-vl-max.', 'code': 404}, 'u...
 Failed! API Response: Error code: 404 - {'error': {'message': 'No endpoints found for qwen/qwen-vl-max.', 'code': 404}, 'user_id': 'user_3BTXKchvH9QzhM8drP410vIMOve'}
  -> gemma_4_31b... OK (16.8s | 522 out / 605 in | 31.1 tok/sec)
  -> llama_4_maverick... OK (2.8s | 89 out / 2196 in | 31.8 tok/sec)
  -> pixtral_large...
    FATAL ERROR (No Retry): Error code: 404 - {'error': {'message': 'No endpoints found for mistralai/pixtral-large-2411.', 'cod...
 Failed! AP

,Master_Case_ID,Associated_Images,Image_SHA256,Diagnosis_gpt_5_5,Likert_gpt_5_5,Prompt_Tokens_gpt_5_5,Total_Tokens_Out_gpt_5_5,Reasoning_Tokens_gpt_5_5,Latency_gpt_5_5,Provider_gpt_5_5,...,Provider_nemotron_3_omni,Timestamp_UTC_nemotron_3_omni,Reasoning_nemotron_3_omni,Reasoning_Raw_nemotron_3_omni,Reasoning_Details_nemotron_3_omni,Actual_Request_Extra_nemotron_3_omni,Grok_Fallback_Used_nemotron_3_omni,OpenRouter_Response_Model_nemotron_3_omni,Usage_JSON_nemotron_3_omni,Raw_Response_nemotron_3_omni
0,1,1.png,6c737472aac9d936,Carotid cavernous fistula,4,1284,543,516,18.2,OpenAI,...,Nvidia,2026-06-12T10:49:53.731580+00:00,The user wants a diagnosis based on the provid...,The user wants a diagnosis based on the provid...,"[{""type"": ""reasoning.text"", ""text"": ""The user ...","{""reasoning"": {""enabled"": true}}",,nvidia/nemotron-3-nano-omni-30b-a3b-reasoning-...,"{""completion_tokens"": 2229, ""prompt_tokens"": 1...","{""diagnosis"":""Glomus jugulare tumor"", ""likert_..."


In [ ]:
# ==========================================
# 5. SCORER'S VIEW & PIVOT
# ==========================================
df_scorer, display_df, scorer_csv = create_scorer_view(final_output_csv)

print(f"Scorer version saved to: {scorer_csv}")
print("\nTRANSPOSED CONSENSUS VIEW:")
display(display_df)
